# Reto 9: Sistema de Gestión de Calificaciones
## Programación para Ciencia de Datos — IPN, Febrero-Julio 2026

In [1]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from datetime import datetime
import json

## Parte 1: Carga y Exploración de Datos

In [2]:
def cargar_datos() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Carga los datos de estudiantes, calificaciones y materias."""
    estudiantes = pd.DataFrame({
        'boleta': ['2021630001','2021630002','2021630003','2021630004','2021630005',
                   '2022630001','2022630002','2022630003','2022630004','2022630005',
                   '2023630001','2023630002','2023630003','2023630004','2023630005'],
        'nombre': ['Juan Pérez García','María López Ruiz','Pedro Sánchez Torres',
                   'Ana Martínez Díaz','Luis Rodríguez Vega','Carmen Flores Luna',
                   'Roberto Díaz Mora','Laura Torres Silva','Diego Ramírez Cruz',
                   'Sofía Vargas Romo','Carlos Mendoza Ríos','Patricia Ortiz León',
                   'Miguel Ángel Castro','Fernanda Reyes Paz','Andrés Guzmán Villa'],
        'semestre': [4,4,4,4,4,3,3,3,3,3,2,2,2,2,2],
        'carrera': ['CD']*15,
        'email': ['juan.perez@ipn.mx','maria.lopez@ipn.mx','pedro.sanchez@ipn.mx',
                  'ana.martinez@ipn.mx','luis.rodriguez@ipn.mx','carmen.flores@ipn.mx',
                  'roberto.diaz@ipn.mx','laura.torres@ipn.mx','diego.ramirez@ipn.mx',
                  'sofia.vargas@ipn.mx','carlos.mendoza@ipn.mx','patricia.ortiz@ipn.mx',
                  'miguel.castro@ipn.mx','fernanda.reyes@ipn.mx','andres.guzman@ipn.mx']
    })
    materias = pd.DataFrame({
        'materia_id': ['MAT101','MAT102','PROG101','PROG102','EST101','EST102','BD101'],
        'nombre': ['Cálculo Diferencial','Cálculo Integral','Programación I',
                   'Programación II','Probabilidad','Estadística Inferencial','Bases de Datos'],
        'creditos': [8,8,6,6,6,6,6],
        'semestre_materia': [1,2,1,2,2,3,3]
    })
    np.random.seed(42)
    calificaciones_data = []
    for boleta in estudiantes['boleta']:
        semestre = estudiantes[estudiantes['boleta'] == boleta]['semestre'].values[0]
        materias_cursadas = materias[materias['semestre_materia'] <= semestre]['materia_id'].tolist()
        for materia in materias_cursadas:
            base  = np.random.uniform(5, 10)
            p1    = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            p2    = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            final = round(min(10, max(0, base + np.random.normal(0, 0.5))), 1)
            if np.random.random() < 0.05:
                p2 = np.nan
            calificaciones_data.append({'boleta': boleta, 'materia_id': materia,
                                        'parcial_1': p1, 'parcial_2': p2, 'final': final})
    return estudiantes, pd.DataFrame(calificaciones_data), materias

In [3]:
def info_general(df_estudiantes: pd.DataFrame, df_calificaciones: pd.DataFrame) -> Dict:
    """Genera información general del sistema."""
    return {
        "total_estudiantes":     len(df_estudiantes),
        "total_registros_calif": len(df_calificaciones),
        "semestres":              sorted(df_estudiantes['semestre'].unique().tolist()),
        "materias_con_registros": df_calificaciones['materia_id'].nunique()
    }

In [4]:
def validar_datos(df_calificaciones: pd.DataFrame) -> Dict:
    """Valida la integridad de los datos."""
    cols_calif = ['parcial_1', 'parcial_2', 'final']
    # Filas con al menos un NaN
    registros_con_nulos = int(df_calificaciones[cols_calif].isna().any(axis=1).sum())
    # Valores fuera del rango [0, 10] — stack() descarta NaN automáticamente
    vals = df_calificaciones[cols_calif].stack()
    fuera_rango = int(((vals < 0) | (vals > 10)).sum())
    return {
        "registros_con_nulos":        registros_con_nulos,
        "calificaciones_fuera_rango": fuera_rango,
        "datos_validos":              (registros_con_nulos == 0) and (fuera_rango == 0)
    }

In [5]:
# Cargar y explorar datos
df_estudiantes, df_calificaciones, df_materias = cargar_datos()

print(f"Estudiantes ({len(df_estudiantes)} registros):")
display(df_estudiantes.head(3))

print(f"\nCalificaciones ({len(df_calificaciones)} registros):")
display(df_calificaciones.head(5))

print(f"\nMaterias ({len(df_materias)} registros):")
display(df_materias)

print("\n── Información general ──")
print(info_general(df_estudiantes, df_calificaciones))

print("\n── Validación de datos ──")
print(validar_datos(df_calificaciones))

Estudiantes (15 registros):


,boleta,nombre,semestre,carrera,email
0,2021630001,Juan Pérez García,4,CD,juan.perez@ipn.mx
1,2021630002,María López Ruiz,4,CD,maria.lopez@ipn.mx
2,2021630003,Pedro Sánchez Torres,4,CD,pedro.sanchez@ipn.mx



Calificaciones (95 registros):


,boleta,materia_id,parcial_1,parcial_2,final
0,2021630001,MAT101,5.8,7.2,7.0
1,2021630001,MAT102,6.1,4.5,4.8
2,2021630001,PROG101,3.9,7.5,6.9
3,2021630001,PROG102,4.9,5.8,5.4
4,2021630001,EST101,8.6,6.4,6.1



Materias (7 registros):


,materia_id,nombre,creditos,semestre_materia
0,MAT101,Cálculo Diferencial,8,1
1,MAT102,Cálculo Integral,8,2
2,PROG101,Programación I,6,1
3,PROG102,Programación II,6,2
4,EST101,Probabilidad,6,2
5,EST102,Estadística Inferencial,6,3
6,BD101,Bases de Datos,6,3



── Información general ──
{'total_estudiantes': 15, 'total_registros_calif': 95, 'semestres': [2, 3, 4], 'materias_con_registros': 7}

── Validación de datos ──
{'registros_con_nulos': 5, 'calificaciones_fuera_rango': 0, 'datos_validos': False}


## Parte 2: Consultas y Filtros

In [6]:
def buscar_estudiante(df_estudiantes: pd.DataFrame, criterio: str, valor: str) -> pd.DataFrame:
    """Busca estudiantes por boleta (exacto), nombre (parcial) o semestre."""
    if criterio == 'boleta':
        return df_estudiantes[df_estudiantes['boleta'] == valor]
    elif criterio == 'nombre':
        return df_estudiantes[df_estudiantes['nombre'].str.contains(valor, case=False, na=False)]
    elif criterio == 'semestre':
        return df_estudiantes[df_estudiantes['semestre'] == int(valor)]
    else:
        print(f"Criterio '{criterio}' no reconocido. Usa: boleta, nombre o semestre.")
        return pd.DataFrame()

In [7]:
def obtener_kardex(boleta: str, df_estudiantes: pd.DataFrame,
                   df_calificaciones: pd.DataFrame, df_materias: pd.DataFrame) -> Dict:
    """Obtiene el kardex completo de un estudiante."""
    resultado = {"estudiante": None, "materias": None, "promedio_general": None,
                 "creditos_cursados": None, "materias_aprobadas": None, "materias_reprobadas": None}

    fila_est = df_estudiantes[df_estudiantes['boleta'] == boleta]
    if fila_est.empty:
        return resultado
    resultado['estudiante'] = fila_est.iloc[0].to_dict()

    calif_est = df_calificaciones[df_calificaciones['boleta'] == boleta].copy()
    if calif_est.empty:
        return resultado

    # Promedio por materia (NaN ignorado con mean)
    calif_est['promedio'] = calif_est[['parcial_1','parcial_2','final']].mean(axis=1).round(2)

    # Unir con información de materias
    tabla = calif_est.merge(df_materias[['materia_id','nombre','creditos']], on='materia_id', how='left')
    tabla = tabla[['materia_id','nombre','parcial_1','parcial_2','final','promedio','creditos']]
    tabla.columns = ['ID','Materia','Parcial 1','Parcial 2','Final','Promedio','Créditos']

    resultado['materias']           = tabla.reset_index(drop=True)
    resultado['promedio_general']   = round(tabla['Promedio'].mean(), 2)
    resultado['creditos_cursados']  = int(tabla['Créditos'].sum())
    resultado['materias_aprobadas'] = int((tabla['Promedio'] >= 6.0).sum())
    resultado['materias_reprobadas']= int((tabla['Promedio'] < 6.0).sum())
    return resultado

In [8]:
def filtrar_por_rendimiento(df_calificaciones: pd.DataFrame,
                            df_estudiantes: pd.DataFrame,
                            min_promedio: float = None,
                            max_promedio: float = None) -> pd.DataFrame:
    """Filtra estudiantes por rango de promedio general."""
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)
    promedios = df.groupby('boleta')['prom_materia'].mean().round(2).reset_index()
    promedios.columns = ['boleta', 'promedio_general']

    if min_promedio is not None:
        promedios = promedios[promedios['promedio_general'] >= min_promedio]
    if max_promedio is not None:
        promedios = promedios[promedios['promedio_general'] <= max_promedio]

    resultado = promedios.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta')
    return resultado[['boleta','nombre','semestre','promedio_general']]\
        .sort_values('promedio_general', ascending=False).reset_index(drop=True)

In [9]:
# Pruebas Parte 2
print("── Buscar por nombre 'María' ──")
display(buscar_estudiante(df_estudiantes, 'nombre', 'María'))

print("\n── Buscar por semestre 3 ──")
display(buscar_estudiante(df_estudiantes, 'semestre', '3'))

print("\n── Kardex 2021630001 ──")
kardex_prueba = obtener_kardex('2021630001', df_estudiantes, df_calificaciones, df_materias)
display(kardex_prueba['materias'])
print(f"Promedio: {kardex_prueba['promedio_general']} | Créditos: {kardex_prueba['creditos_cursados']}")
print(f"Aprobadas: {kardex_prueba['materias_aprobadas']} | Reprobadas: {kardex_prueba['materias_reprobadas']}")

print("\n── Filtrar promedio entre 7.5 y 9.0 ──")
display(filtrar_por_rendimiento(df_calificaciones, df_estudiantes, min_promedio=7.5, max_promedio=9.0))

── Buscar por nombre 'María' ──


,boleta,nombre,semestre,carrera,email
1,2021630002,María López Ruiz,4,CD,maria.lopez@ipn.mx



── Buscar por semestre 3 ──


,boleta,nombre,semestre,carrera,email
5,2022630001,Carmen Flores Luna,3,CD,carmen.flores@ipn.mx
6,2022630002,Roberto Díaz Mora,3,CD,roberto.diaz@ipn.mx
7,2022630003,Laura Torres Silva,3,CD,laura.torres@ipn.mx
8,2022630004,Diego Ramírez Cruz,3,CD,diego.ramirez@ipn.mx
9,2022630005,Sofía Vargas Romo,3,CD,sofia.vargas@ipn.mx



── Kardex 2021630001 ──


,ID,Materia,Parcial 1,Parcial 2,Final,Promedio,Créditos
0,MAT101,Cálculo Diferencial,5.8,7.2,7.0,6.67,8
1,MAT102,Cálculo Integral,6.1,4.5,4.8,5.13,8
2,PROG101,Programación I,3.9,7.5,6.9,6.10,6
3,PROG102,Programación II,4.9,5.8,5.4,5.37,6
4,EST101,Probabilidad,8.6,6.4,6.1,7.03,6
5,EST102,Estadística Inferencial,4.8,4.7,5.8,5.10,6
6,BD101,Bases de Datos,7.4,8.3,8.2,7.97,6


Promedio: 6.2 | Créditos: 46
Aprobadas: 4 | Reprobadas: 3

── Filtrar promedio entre 7.5 y 9.0 ──


,boleta,nombre,semestre,promedio_general
0,2023630003,Miguel Ángel Castro,2,8.91
1,2023630004,Fernanda Reyes Paz,2,8.59
2,2021630005,Luis Rodríguez Vega,4,8.44
3,2023630005,Andrés Guzmán Villa,2,8.43
4,2022630003,Laura Torres Silva,3,7.96
5,2021630004,Ana Martínez Díaz,4,7.90
6,2022630002,Roberto Díaz Mora,3,7.90
7,2022630005,Sofía Vargas Romo,3,7.74
8,2022630001,Carmen Flores Luna,3,7.73


## Parte 3: Cálculos y Estadísticas

In [10]:
def _promedios_por_alumno(df_calificaciones: pd.DataFrame) -> pd.DataFrame:
    """Helper: calcula el promedio general por alumno."""
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)
    return df.groupby('boleta')['prom_materia'].mean().round(2).reset_index(name='promedio_general')

In [11]:
def calcular_promedio_materia(df_calificaciones: pd.DataFrame, materia_id: str) -> Dict:
    """Calcula estadísticas detalladas de una materia."""
    df = df_calificaciones[df_calificaciones['materia_id'] == materia_id].copy()
    if df.empty:
        return {"materia": materia_id, "inscritos": 0}
    df['promedio'] = df[['parcial_1','parcial_2','final']].mean(axis=1)
    return {
        "materia":               materia_id,
        "inscritos":             len(df),
        "promedio_parcial1":     round(df['parcial_1'].mean(), 2),
        "promedio_parcial2":     round(df['parcial_2'].mean(), 2),   # NaN ignorado
        "promedio_final":        round(df['final'].mean(), 2),
        "promedio_general":      round(df['promedio'].mean(), 2),
        "tasa_aprobacion":       round((df['promedio'] >= 6.0).mean() * 100, 1),
        "calificacion_maxima":   round(df['promedio'].max(), 2),
        "calificacion_minima":   round(df['promedio'].min(), 2)
    }

In [12]:
def ranking_estudiantes(df_calificaciones: pd.DataFrame,
                        df_estudiantes: pd.DataFrame,
                        top_n: int = 10) -> pd.DataFrame:
    """Genera ranking de mejores estudiantes por promedio general."""
    promedios = _promedios_por_alumno(df_calificaciones)
    ranking = promedios.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta')
    ranking = ranking.sort_values('promedio_general', ascending=False).head(top_n).reset_index(drop=True)
    ranking.index += 1
    ranking.index.name = 'posicion'
    return ranking[['nombre','semestre','promedio_general']].reset_index()

In [13]:
def estadisticas_por_semestre(df_estudiantes: pd.DataFrame,
                               df_calificaciones: pd.DataFrame) -> pd.DataFrame:
    """Estadísticas académicas agrupadas por semestre."""
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)

    por_alumno = df.groupby('boleta').agg(
        promedio_general = ('prom_materia', 'mean'),
        aprobadas        = ('prom_materia', lambda x: (x >= 6.0).sum()),
        total_materias   = ('prom_materia', 'count')
    ).reset_index()

    df_merge = por_alumno.merge(df_estudiantes[['boleta','semestre']], on='boleta')

    stats = df_merge.groupby('semestre').agg(
        num_estudiantes = ('boleta', 'count'),
        promedio        = ('promedio_general', 'mean'),
        tasa_aprobacion = ('aprobadas',
            lambda x: (x / df_merge.loc[x.index,'total_materias']).mean() * 100),
        mejor_promedio  = ('promedio_general', 'max'),
        peor_promedio   = ('promedio_general', 'min')
    ).round(2)

    stats.columns = ['Estudiantes','Promedio','Tasa Aprob. (%)','Mejor Prom.','Peor Prom.']
    return stats

In [14]:
# Pruebas Parte 3
print("── Estadísticas MAT101 ──")
print(calcular_promedio_materia(df_calificaciones, 'MAT101'))

print("\n── Ranking Top 5 ──")
display(ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5))

print("\n── Estadísticas por semestre ──")
display(estadisticas_por_semestre(df_estudiantes, df_calificaciones))

── Estadísticas MAT101 ──
{'materia': 'MAT101', 'inscritos': 15, 'promedio_parcial1': np.float64(7.76), 'promedio_parcial2': np.float64(7.91), 'promedio_final': np.float64(7.82), 'promedio_general': np.float64(7.79), 'tasa_aprobacion': np.float64(100.0), 'calificacion_maxima': np.float64(9.6), 'calificacion_minima': np.float64(6.0)}

── Ranking Top 5 ──


,posicion,nombre,semestre,promedio_general
0,1,Miguel Ángel Castro,2,8.91
1,2,Fernanda Reyes Paz,2,8.59
2,3,Luis Rodríguez Vega,4,8.44
3,4,Andrés Guzmán Villa,2,8.43
4,5,Laura Torres Silva,3,7.96



── Estadísticas por semestre ──


,Estudiantes,Promedio,Tasa Aprob. (%),Mejor Prom.,Peor Prom.
semestre,,,,,
2,5,7.96,92.00,8.91,6.46
3,5,7.75,91.43,7.96,7.41
4,5,7.30,85.71,8.44,6.20


## Parte 4: Identificación de Riesgo y Reportes

In [15]:
def identificar_estudiantes_riesgo(df_calificaciones: pd.DataFrame,
                                    df_estudiantes: pd.DataFrame,
                                    umbral_promedio: float = 7.0,
                                    max_reprobadas: int = 2) -> pd.DataFrame:
    """Identifica estudiantes en riesgo académico con clasificación de motivo."""
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)

    por_alumno = df.groupby('boleta').agg(
        promedio_general = ('prom_materia', 'mean'),
        reprobadas       = ('prom_materia', lambda x: (x < 6.0).sum())
    ).reset_index()
    por_alumno['promedio_general'] = por_alumno['promedio_general'].round(2)

    bajo_prom  = por_alumno['promedio_general'] < umbral_promedio
    muchas_rep = por_alumno['reprobadas'] > max_reprobadas

    en_riesgo = por_alumno[bajo_prom | muchas_rep].copy()

    def motivo(row):
        b = row['promedio_general'] < umbral_promedio
        r = row['reprobadas'] > max_reprobadas
        if b and r:  return "Ambos"
        if b:        return "Bajo promedio"
        return "Materias reprobadas"

    en_riesgo['motivo'] = en_riesgo.apply(motivo, axis=1)

    resultado = en_riesgo.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta')
    return resultado[['boleta','nombre','semestre','promedio_general','reprobadas','motivo']]\
        .sort_values('promedio_general').reset_index(drop=True)

In [16]:
def generar_reporte_academico(df_estudiantes: pd.DataFrame,
                               df_calificaciones: pd.DataFrame,
                               df_materias: pd.DataFrame) -> Dict:
    """Genera reporte académico completo integrando todas las funciones."""
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)

    # Estadísticas por materia
    por_materia = df.groupby('materia_id').agg(
        inscritos       = ('boleta','count'),
        promedio        = ('prom_materia','mean'),
        tasa_aprobacion = ('prom_materia', lambda x: round((x>=6.0).mean()*100, 1)),
        max_calif       = ('prom_materia','max'),
        min_calif       = ('prom_materia','min')
    ).round(2).reset_index()
    por_materia = por_materia.merge(df_materias[['materia_id','nombre']], on='materia_id')
    por_materia = por_materia[['materia_id','nombre','inscritos','promedio',
                               'tasa_aprobacion','max_calif','min_calif']]

    return {
        "resumen_general": {
            "total_estudiantes": len(df_estudiantes),
            "promedio_global":   round(df['prom_materia'].mean(), 2),
            "tasa_aprobacion":   round((df['prom_materia'] >= 6.0).mean() * 100, 1)
        },
        "por_semestre":      estadisticas_por_semestre(df_estudiantes, df_calificaciones),
        "por_materia":       por_materia,
        "mejores_estudiantes": ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=10),
        "estudiantes_riesgo":  identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes),
        "fecha_generacion":    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

In [17]:
def exportar_kardex(boleta: str, kardex: Dict, formato: str = 'csv') -> str:
    """Exporta el kardex de un estudiante a CSV o JSON."""
    fecha = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre_archivo = f"kardex_{boleta}_{fecha}.{formato}"

    if formato == 'csv':
        kardex['materias'].to_csv(nombre_archivo, index=False)
    elif formato == 'json':
        datos = {
            "estudiante":        kardex['estudiante'],
            "promedio_general":  kardex['promedio_general'],
            "creditos_cursados": kardex['creditos_cursados'],
            "materias_aprobadas":  kardex['materias_aprobadas'],
            "materias_reprobadas": kardex['materias_reprobadas'],
            "materias":          kardex['materias'].to_dict(orient='records')
        }
        with open(nombre_archivo, 'w', encoding='utf-8') as f:
            json.dump(datos, f, ensure_ascii=False, indent=2)
    else:
        print(f"Formato '{formato}' no soportado. Usa 'csv' o 'json'.")
        return ""

    print(f"✅ Archivo generado: {nombre_archivo}")
    return nombre_archivo

## Visualizadores (provistos por el enunciado)

In [18]:
def mostrar_kardex(kardex: Dict) -> None:
    """Muestra el kardex de forma legible."""
    if kardex['estudiante'] is None:
        print("❌ Estudiante no encontrado"); return
    est = kardex['estudiante']
    print("="*70)
    print("                         KARDEX ACADÉMICO")
    print("="*70)
    print(f"\n📋 DATOS DEL ESTUDIANTE\n" + "-"*40)
    print(f"Boleta: {est.get('boleta','N/A')}")
    print(f"Nombre: {est.get('nombre','N/A')}")
    print(f"Semestre: {est.get('semestre','N/A')}")
    print(f"Carrera: {est.get('carrera','N/A')}")
    print(f"Email: {est.get('email','N/A')}")
    print(f"\n📚 CALIFICACIONES\n" + "-"*70)
    if kardex['materias'] is not None and len(kardex['materias']) > 0:
        print(kardex['materias'].to_string(index=False))
    else:
        print("Sin calificaciones registradas")
    print(f"\n📊 RESUMEN\n" + "-"*40)
    print(f"Promedio General:    {kardex.get('promedio_general',0):.2f}")
    print(f"Créditos Cursados:   {kardex.get('creditos_cursados',0)}")
    print(f"Materias Aprobadas:  {kardex.get('materias_aprobadas',0)}")
    print(f"Materias Reprobadas: {kardex.get('materias_reprobadas',0)}")
    print("="*70)

def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte académico completo."""
    print("="*70)
    print("              REPORTE ACADÉMICO - CIENCIA DE DATOS")
    print(f"              Generado: {reporte['fecha_generacion']}")
    print("="*70)
    res = reporte.get('resumen_general', {})
    print(f"\n📊 RESUMEN GENERAL\n" + "-"*40)
    print(f"Total de estudiantes: {res.get('total_estudiantes','N/A')}")
    print(f"Promedio global:      {res.get('promedio_global',0):.2f}")
    print(f"Tasa de aprobación:   {res.get('tasa_aprobacion',0):.1f}%")
    if reporte.get('por_semestre') is not None:
        print(f"\n📅 ESTADÍSTICAS POR SEMESTRE\n" + "-"*40)
        print(reporte['por_semestre'].to_string())
    if reporte.get('mejores_estudiantes') is not None:
        print(f"\n🏆 TOP 5 ESTUDIANTES\n" + "-"*40)
        print(reporte['mejores_estudiantes'].head().to_string(index=False))
    riesgo = reporte.get('estudiantes_riesgo')
    if riesgo is not None and len(riesgo) > 0:
        print(f"\n⚠️  ESTUDIANTES EN RIESGO ({len(riesgo)})\n" + "-"*40)
        print(riesgo.to_string(index=False))
    else:
        print("\n✅ No hay estudiantes en riesgo académico")
    print("\n" + "="*70)

In [19]:
# Pruebas Parte 4
print("── Estudiantes en riesgo (umbral=7.0, max_reprobadas=2) ──")
display(identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes))

print("\n── Reporte académico completo ──")
reporte = generar_reporte_academico(df_estudiantes, df_calificaciones, df_materias)
mostrar_reporte(reporte)

print("\n── Kardex completo y exportación ──")
kardex = obtener_kardex('2021630001', df_estudiantes, df_calificaciones, df_materias)
mostrar_kardex(kardex)

archivo_csv  = exportar_kardex('2021630001', kardex, 'csv')
archivo_json = exportar_kardex('2021630001', kardex, 'json')

── Estudiantes en riesgo (umbral=7.0, max_reprobadas=2) ──


,boleta,nombre,semestre,promedio_general,reprobadas,motivo
0,2021630001,Juan Pérez García,4,6.20,3,Ambos
1,2023630002,Patricia Ortiz León,2,6.46,1,Bajo promedio
2,2021630002,María López Ruiz,4,6.86,0,Bajo promedio



── Reporte académico completo ──
              REPORTE ACADÉMICO - CIENCIA DE DATOS
              Generado: 2026-06-23 16:26:01

📊 RESUMEN GENERAL
----------------------------------------
Total de estudiantes: 15
Promedio global:      7.64
Tasa de aprobación:   89.5%

📅 ESTADÍSTICAS POR SEMESTRE
----------------------------------------
          Estudiantes  Promedio  Tasa Aprob. (%)  Mejor Prom.  Peor Prom.
semestre                                                                 
2                   5      7.96            92.00         8.91        6.46
3                   5      7.75            91.43         7.96        7.41
4                   5      7.30            85.71         8.44        6.20

🏆 TOP 5 ESTUDIANTES
----------------------------------------
 posicion              nombre  semestre  promedio_general
        1 Miguel Ángel Castro         2              8.91
        2  Fernanda Reyes Paz         2              8.59
        3 Luis Rodríguez Vega         4              8.

✅ Archivo generado: kardex_2021630001_20260623_162601.json


## Bonus: Predictor de riesgo y Comparador de rendimiento

In [20]:
def predecir_riesgo_proximo_semestre(df_calificaciones: pd.DataFrame,
                                      df_estudiantes: pd.DataFrame) -> pd.DataFrame:
    """
    Predice alumnos en riesgo el próximo semestre basándose en tendencias:
    - Tendencia decreciente: parcial_2 < parcial_1
    - Caída al final: final < parcial_1
    Se considera en riesgo si tiene ≥2 materias con alguna de esas señales.
    """
    df = df_calificaciones.copy()

    # Tendencia decreciente de parcial 1 a parcial 2
    df['tend_dec_12']  = df['parcial_2'] < df['parcial_1']         # True si baja
    # Caída respecto al primer parcial hasta el final
    df['tend_dec_fin'] = df['final'] < df['parcial_1']             # True si cae al final
    df['señal_riesgo'] = df['tend_dec_12'] | df['tend_dec_fin']

    por_alumno = df.groupby('boleta').agg(
        materias_con_tendencia = ('señal_riesgo', 'sum'),
        total_materias         = ('señal_riesgo', 'count')
    ).reset_index()

    por_alumno['pct_tendencia'] = (por_alumno['materias_con_tendencia'] /
                                   por_alumno['total_materias'] * 100).round(1)

    en_riesgo = por_alumno[por_alumno['materias_con_tendencia'] >= 2].copy()
    resultado = en_riesgo.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta')
    return resultado[['boleta','nombre','semestre','materias_con_tendencia','pct_tendencia']]\
        .sort_values('pct_tendencia', ascending=False).reset_index(drop=True)

In [21]:
def comparar_estudiantes(boleta1: str, boleta2: str,
                          df_calificaciones: pd.DataFrame,
                          df_estudiantes: pd.DataFrame,
                          df_materias: pd.DataFrame) -> Dict:
    """Compara el rendimiento de dos estudiantes en sus materias compartidas."""
    k1 = obtener_kardex(boleta1, df_estudiantes, df_calificaciones, df_materias)
    k2 = obtener_kardex(boleta2, df_estudiantes, df_calificaciones, df_materias)

    if k1['estudiante'] is None or k2['estudiante'] is None:
        return {"error": "Uno o ambos estudiantes no encontrados"}

    # Materias en común
    ids1 = set(k1['materias']['ID'])
    ids2 = set(k2['materias']['ID'])
    comunes = ids1 & ids2

    comparacion = None
    if comunes:
        m1 = k1['materias'][k1['materias']['ID'].isin(comunes)][['ID','Materia','Promedio']]\
               .rename(columns={'Promedio': f"Prom_{k1['estudiante']['nombre'].split()[0]}"})
        m2 = k2['materias'][k2['materias']['ID'].isin(comunes)][['ID','Promedio']]\
               .rename(columns={'Promedio': f"Prom_{k2['estudiante']['nombre'].split()[0]}"})
        comparacion = m1.merge(m2, on='ID').reset_index(drop=True)

    return {
        "estudiante_1": {
            "nombre":           k1['estudiante']['nombre'],
            "semestre":         k1['estudiante']['semestre'],
            "promedio_general": k1['promedio_general'],
            "aprobadas":        k1['materias_aprobadas'],
            "reprobadas":       k1['materias_reprobadas']
        },
        "estudiante_2": {
            "nombre":           k2['estudiante']['nombre'],
            "semestre":         k2['estudiante']['semestre'],
            "promedio_general": k2['promedio_general'],
            "aprobadas":        k2['materias_aprobadas'],
            "reprobadas":       k2['materias_reprobadas']
        },
        "materias_comunes": comparacion
    }

In [22]:
# Pruebas Bonus
print("── Predictor de riesgo próximo semestre ──")
display(predecir_riesgo_proximo_semestre(df_calificaciones, df_estudiantes))

print("\n── Comparar estudiantes 2021630001 vs 2021630004 ──")
comp = comparar_estudiantes('2021630001','2021630004',
                             df_calificaciones, df_estudiantes, df_materias)
print(f"\n{comp['estudiante_1']['nombre']}  →  Promedio: {comp['estudiante_1']['promedio_general']}")
print(f"{comp['estudiante_2']['nombre']}  →  Promedio: {comp['estudiante_2']['promedio_general']}")
print("\nMaterias en común:")
display(comp['materias_comunes'])

── Predictor de riesgo próximo semestre ──


,boleta,nombre,semestre,materias_con_tendencia,pct_tendencia
0,2022630005,Sofía Vargas Romo,3,7,100.0
1,2022630001,Carmen Flores Luna,3,6,85.7
2,2021630003,Pedro Sánchez Torres,4,6,85.7
3,2023630002,Patricia Ortiz León,2,4,80.0
4,2023630005,Andrés Guzmán Villa,2,4,80.0
5,2023630001,Carlos Mendoza Ríos,2,4,80.0
6,2021630002,María López Ruiz,4,5,71.4
7,2022630004,Diego Ramírez Cruz,3,5,71.4
8,2023630003,Miguel Ángel Castro,2,3,60.0
9,2021630004,Ana Martínez Díaz,4,4,57.1



── Comparar estudiantes 2021630001 vs 2021630004 ──

Juan Pérez García  →  Promedio: 6.2
Ana Martínez Díaz  →  Promedio: 7.9

Materias en común:


,ID,Materia,Prom_Juan,Prom_Ana
0,MAT101,Cálculo Diferencial,6.67,9.33
1,MAT102,Cálculo Integral,5.13,6.55
2,PROG101,Programación I,6.10,7.50
3,PROG102,Programación II,5.37,9.27
4,EST101,Probabilidad,7.03,8.23
5,EST102,Estadística Inferencial,5.10,7.60
6,BD101,Bases de Datos,7.97,6.85
